# DreamGrasp calibration analysis

All logic lives in `dreamgrasp.eval.correlate` (no notebook-only logic — engineering rule).
This notebook is a thin interactive view: synthetic self-check, then the real parquets when they exist.

In [ ]:
from pathlib import Path
import pandas as pd
from dreamgrasp.eval.correlate import (
    make_synthetic, ranking_reliability, task_level_pearson, trust_region_chart, REPO_ROOT
)

In [ ]:
# Synthetic sanity: the pipeline must recover a known correlation.
for target in (0.0, 0.5, 1.0):
    sim, dream = make_synthetic(target)
    rho, lo, hi = ranking_reliability(sim, dream)['synthetic']
    print(f'target~{target}: rho={rho:.3f} CI=[{lo:.3f},{hi:.3f}]')

In [ ]:
# Real results (Type 2): per-tier reliability, in-distribution vs held-out, trust region.
sim_p = REPO_ROOT / 'results' / 'sim_success.parquet'
dream_p = REPO_ROOT / 'results' / 'dream_success.parquet'
fid_p = REPO_ROOT / 'results' / 'wm_fidelity.parquet'
if sim_p.exists() and dream_p.exists():
    sim, dream = pd.read_parquet(sim_p), pd.read_parquet(dream_p)
    rel = ranking_reliability(sim, dream)
    display(pd.DataFrame(rel, index=['rho', 'lo', 'hi']).T)
    if fid_p.exists():
        fdf = pd.read_parquet(fid_p)
        fid = {Path(c).name: float(g['divergence_step'].mean()) for c, g in fdf.groupby('checkpoint')}
        fig = trust_region_chart(rel, fid, REPO_ROOT / 'report' / 'figures' / 'trust_region.png')
else:
    print('real parquets not present yet (Type 2)')